# Hybrid full first3 of50 rounds
100%train80%, fulltest20% everyround, descriptive only. AlllearnedmodelsDAGSNet, seed42. Compiled-vs-eager real-data numerical gate beforetrain; one arm/GPU. No rawrecords exported; prepareddata only/kaggle/temp. W&Bonline/privateinlinekey owner-authorized. Checkpoint willresume fixed50round schedule after actualbudget review. R1/R2notproved.

In [1]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.1 which is incompatible.
wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] 

W&B authenticated online


In [2]:
CODE=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'model_packet.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'teacher_base.py', 'teacher_driver.py', 'v10_energy.py', 'v10_run.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py', 'v5_anchor.py', 'v5_books.py', 'v5_run.py', 'v6_geometry.py', 'v6_run.py', 'v6_schema.py', 'v7_basis.py', 'v7_nystrom.py', 'v7_run.py', 'v8_hybrid.py', 'v8_run.py', 'v9_anchor.py', 'v9_run.py']
PROJ=['full_driver.py', 'full_method.py', 'full_prepare.py', 'full_run.py', 'full_verify.py', 'parent_run.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
os.environ['TORCHINDUCTOR_COMPILE_THREADS']='1'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
print('torch',torch.__version__,'gpus',[torch.cuda.get_device_name(i) for i in range(2)],flush=True)
hits=[p.parent for p in Path('/kaggle/input').rglob('full_run.py') if (p.parent/'source_manifest.json').exists()]
assert len(hits)==1,hits
SRC=hits[0];WORK=Path('/kaggle/temp/full-context');OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source');SOURCE.mkdir(exist_ok=True)
for name in ('mh','knowledge','papers/hybrid-full/proj'):(WORK/name).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
    target=WORK/'mh'/name if name in CODE else WORK/'papers/hybrid-full/proj'/name if name in PROJ else WORK/'knowledge'/name
    shutil.copyfile(SRC/name,target)
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'papers/hybrid-full/proj'))
from full_prepare import prepare_full
data=prepare_full(WORK)
(SOURCE/'data_manifest.json').write_text(json.dumps(data,indent=2))
print('verified full source and data',data['source_rows'],data['test_rows'],flush=True)


torch 2.10.0+cu128 gpus ['Tesla T4', 'Tesla T4']
Prepared real pilot {'train_0': 3352748, 'train_1': 1783872, 'train_2': 2652819, 'train_3': 1455566, 'train_4': 1864534, 'train_5': 1932238, 'train_6': 3372587, 'train_7': 1472203, 'train_8': 2845626, 'train_9': 1240841, 'train_10': 2671366, 'train_11': 1851046, 'train_12': 1495960, 'train_13': 1779645, 'train_14': 3063594, 'train_15': 1212957, 'train_16': 870217, 'train_17': 5890990, 'train_18': 1063458, 'train_19': 1173148} old cut: 0
verified full source and data 43045415 10761343


In [3]:
METHODS=['h_local', 'h_dml']

OUT.mkdir(exist_ok=True);active={};summary={}
for gpu,arm in enumerate(METHODS):
    folder=OUT/f'{arm}_s42';folder.mkdir(exist_ok=True)
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
    command=[sys.executable,str(WORK/'papers/hybrid-full/proj/full_run.py'),
             '--arm',arm,'--seed','42','--rounds','50','--eval_every','1','--backend','compiled',
             '--wandb_online','--wandb_project','mobihet-hybrid-full','--out',str(OUT),
             '--stop_after','3','--max_hours','4.2']
    log=open(folder/'log.txt','w');process=subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
    active[gpu]=(arm,process,log)
while active:
    for gpu,(arm,process,log) in list(active.items()):
        if process.poll() is None and time.monotonic()-SESSION_T0>=4.8*3600:process.terminate()
        if process.poll() is None and time.monotonic()-SESSION_T0>=4.85*3600:process.kill()
        if process.poll() is None:continue
        log.close();del active[gpu]
        path=OUT/f'{arm}_s42/result.json';result=json.loads(path.read_text()) if path.exists() else {}
        rows=result.get('history',[])
        summary[arm]=dict(rc=process.returncode,complete=result.get('complete',False),
                         calibration_complete=len(rows)==3,rounds=len(rows),target_rounds=50)
        if rows:
            worst=max(row['sec'] for row in rows)
            setup=max(0,result['minutes']*60-sum(row['sec'] for row in rows))
            estimate=1.15*(setup+worst*(50-len(rows)))/3600
            summary[arm].update(worst_round_seconds=worst,setup_seconds=setup,
                conservative_remaining_gpu_hours=estimate,
                note='one GPU per arm; Kaggle account/session cost must be measured separately')
    time.sleep(3)
(OUT/'summary_s42.json').write_text(json.dumps(summary,indent=2))
print(summary,flush=True)
assert all(v['rc']==0 and v['calibration_complete'] for v in summary.values()),'incomplete first3 full rounds; preserve partial'


{'h_dml': {'rc': 0, 'complete': False, 'calibration_complete': False, 'rounds': 2, 'target_rounds': 50, 'worst_round_seconds': 6648.6314198970795, 'setup_seconds': 51.189234256744385, 'conservative_remaining_gpu_hours': 101.96203388825388, 'note': 'one GPU per arm; Kaggle account/session cost must be measured separately'}, 'h_local': {'rc': 0, 'complete': False, 'calibration_complete': False, 'rounds': 2, 'target_rounds': 50, 'worst_round_seconds': 6652.05893778801, 'setup_seconds': 51.130372762680054, 'conservative_remaining_gpu_hours': 102.01457035960422, 'note': 'one GPU per arm; Kaggle account/session cost must be measured separately'}}


AssertionError: incomplete first3 full rounds; preserve partial